In [1]:
from netgen.occ import *
from netgen.webgui import Draw as DrawGeo
import ngsolve

In [2]:
box = Box((0,0,0), (3,0.6,1))
box.faces.name="outer"
cyl = sum( [Cylinder((0.5+i,0,0.5), Y, 0.25,0.8) for i in range(3)] )
cyl.faces.name="cyl"
geo = box-cyl

DrawGeo(geo);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': …

In [3]:
cylboxedges = geo.faces["outer"].edges * geo.faces["cyl"].edges
cylboxedges.name = "cylbox"
geo = geo.MakeChamfer(cylboxedges, 0.03)

In [4]:
geo.faces.Min(X).name = "fix"
geo.faces.Max(X).name = "force"

DrawGeo(geo);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': …

In [5]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(OCCGeometry(geo).GenerateMesh(maxh=0.1)).Curve(3)
Draw (mesh);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [6]:
E, nu = 210, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)

In [7]:
fes = VectorH1(mesh, order=3, dirichlet="fix")
u,v = fes.TnT()
gfu = GridFunction(fes)

with TaskManager():
    a = BilinearForm(InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v))).Compile()*dx)
    pre = Preconditioner(a, "bddc")
    a.Assemble()

In [8]:
force = CF( (1e-3,0,0) )
f = LinearForm(force*v*ds("force")).Assemble()

In [9]:
from ngsolve.krylovspace import CGSolver
inv = CGSolver(a.mat, pre, printrates=True, tol=1e-8)
gfu.vec.data = inv * f.vec

CG iteration 1, residual = 0.00017988679254465906     
CG iteration 2, residual = 7.654199381934503e-05     
CG iteration 3, residual = 9.402559286998864e-05     
CG iteration 4, residual = 6.973819265603654e-05     
CG iteration 5, residual = 6.622870430054913e-05     
CG iteration 6, residual = 5.185609125915441e-05     
CG iteration 7, residual = 4.0034054732135683e-05     
CG iteration 8, residual = 3.312000282517064e-05     
CG iteration 9, residual = 2.772676985398685e-05     
CG iteration 10, residual = 1.8339065298926654e-05     
CG iteration 11, residual = 1.4785624697262083e-05     
CG iteration 12, residual = 1.0241153744498507e-05     
CG iteration 13, residual = 7.858556821052018e-06     
CG iteration 14, residual = 6.100554451548121e-06     
CG iteration 15, residual = 4.385062379135019e-06     
CG iteration 16, residual = 3.299669825847753e-06     
CG iteration 17, residual = 2.6067319332263095e-06     
CG iteration 18, residual = 1.8243866055053334e-06     
CG iteration

In [10]:
with TaskManager():
    fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
    gfstress = GridFunction(fesstress)
    gfstress.Interpolate (Stress(Sym(Grad(gfu))))

In [11]:
Draw (5e4*gfu, mesh);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [12]:
Draw (Norm(gfstress), mesh, deformation=1e4*gfu, draw_vol=False, order=3);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…